03_Health_Evaluation.ipynb (v4 — Shape-Template Margin Loss + Per-View Thresholds)

Automatically generated by Colab.

# 🌿 VedaVision — Module 2
# Notebook 03 — Health Evaluation Pipeline (v4)

---
### What changed from v3

**1. Margin loss: reference shape template instead of a single hull ratio**

v3 estimated margin loss with one number (leaf area / convex hull area)
compared to a species baseline. That's a rough proxy -- it can't tell
"this exact notch is missing" from "leaves of this species are just a
bit concave here naturally" as precisely as actually overlaying the
damaged leaf onto what a healthy leaf normally looks like.

v4 builds a real reference shape per species:
  - Every healthy calibration leaf mask (both top AND bottom photos --
    the physical outline is the same leaf either way, so pooling both
    views just gives more data for the same shape) is aligned to a
    common canonical pose: PCA rotation to make the leaf vertical, tip
    pointing up, rescaled to a common length.
  - The aligned masks are averaged into a probability template: for
    each canvas pixel, what fraction of healthy reference leaves had
    tissue there.
  - A query leaf is aligned the same way, then scored by how much of
    the template's "usually-leaf" area is actually present. Missing
    area = margin loss.
  - Falls back to the old hull-ratio heuristic automatically for any
    species you haven't calibrated a shape template for yet.

**2. Colour thresholds are now calibrated PER VIEW (top vs bottom), not
just per species**

The adaxial (top) and abaxial (bottom) leaf surfaces genuinely look
different -- bottom is often paler, duller, sometimes fuzzy, less waxy.
One shared colour threshold risks flagging a healthy pale underside as
damage, using thresholds tuned on the glossier top side -- the same
class of mistake as the original Diya Na bug, one level down. So
`SPECIES_THRESHOLD_OVERRIDES` is now nested `{species: {'top': {...},
'bottom': {...}}}`, and `evaluate_full_leaf_health` automatically passes
the correct view when scoring each photo.

Shape templates, by contrast, are NOT per-view -- the leaf's edge outline
doesn't change depending on which side faces the camera, so combining
top+bottom calibration photos for shape just means more usable data.

---
### Calibration folder structure (updated):
```
health_calibration/species_name/healthy/top/*.jpg
health_calibration/species_name/healthy/bottom/*.jpg
health_calibration/species_name/damaged/top/*.jpg      (optional)
health_calibration/species_name/damaged/bottom/*.jpg   (optional)
```

---
## Cell 1 — Install Libraries

In [ ]:
!pip install opencv-python-headless scikit-image matplotlib numpy tqdm pandas -q
print('✅ Libraries installed!')

## Cell 2 — Import Libraries

In [1]:
import os
import re
import base64
import cv2
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import random
import gc
from glob import glob
from tqdm import tqdm

In [2]:
print('✅ All libraries imported!')

✅ All libraries imported!


## Cell 3 — Mount Google Drive & Configure Paths

In [3]:
HEALTH_RAW_PATH         = 'C:/Users/User/Documents/UOM/L4S2/Research/health/health_raw'
HEALTH_CALIBRATION_PATH = 'C:/Users/User/Documents/UOM/L4S2/Research/health/health_calibration'
HEALTH_RESULTS_PATH     = 'C:/Users/User/Documents/UOM/L4S2/Research/health/health_results'
LOGS_PATH               = 'C:/Users/User/Documents/UOM/L4S2/Research/logs'

In [4]:
VIEWS     = ['top', 'bottom']
WORK_SIZE = (512, 512)

In [5]:
# --------------------------------------------------------------------------
# Colour-quality thresholds. Structure: {species: {'top': {...}, 'bottom': {...}}}
# Only include keys that differ from DEFAULT_THRESHOLDS -- get_thresholds()
# fills in the rest automatically.
# --------------------------------------------------------------------------
DEFAULT_THRESHOLDS = {
    'HEALTHY_HUE_RANGE': (35, 85),
    'HEALTHY_MIN_SAT': 40,
    'HEALTHY_MIN_VAL': 40,
    'YELLOW_HUE_RANGE': (20, 35),
    'YELLOW_MIN_SAT': 40,
    'BROWN_HUE_RANGE': (5, 25),
    'BROWN_MAX_VAL': 150,
    'BLACK_MAX_VAL': 60,
    'BLACK_MAX_SAT': 60,
    'PALE_MAX_SAT': 30,
    'PALE_MIN_VAL': 200,
}

In [6]:
MIN_DAMAGE_BLOB_AREA = 15

In [7]:
SPECIES_THRESHOLD_OVERRIDES = {
    # 'Diya Na': {
    #     'top':    {'HEALTHY_MIN_VAL': 22, 'HEALTHY_MIN_SAT': 25, 'BLACK_MAX_SAT': 55},
    #     'bottom': {'HEALTHY_MIN_VAL': 30, 'HEALTHY_MIN_SAT': 20},
    # },
}

In [8]:
# --------------------------------------------------------------------------
# Structural damage config
# --------------------------------------------------------------------------
BG_COLOR_TOLERANCE = 35
HOLE_BRIDGE_BREAK_PX = 5

In [9]:
# Shape-template alignment canvas (used for both building templates and
# scoring query leaves against them -- must match between the two)
SHAPE_CANVAS_SIZE = 400
SHAPE_TARGET_LENGTH = 340

In [10]:
# Fallback hull-ratio heuristic, used ONLY for species with no shape
# template calibrated yet.
DEFAULT_HULL_RATIO_BASELINE = 0.88
SPECIES_HULL_BASELINE = {
    # 'Kuringchan': 0.79,
}

In [11]:
# Per-species shape templates, stored as base64-encoded PNG (grayscale
# probability map, 0-255). Populated by calibrate_species() -> Cell 14 export.
SPECIES_SHAPE_TEMPLATES_B64 = {
    # 'Diya Na': 'iVBORw0KGgoAAAANSUhEUgAA...'
}

In [12]:
print('✅ Config set!')
os.makedirs(HEALTH_RESULTS_PATH, exist_ok=True)
os.makedirs(LOGS_PATH, exist_ok=True)

✅ Config set!


In [13]:
species_list = sorted([
    d for d in os.listdir(HEALTH_RAW_PATH)
    if os.path.isdir(os.path.join(HEALTH_RAW_PATH, d)) and not d.startswith('.')
]) if os.path.exists(HEALTH_RAW_PATH) else []

In [14]:
print(f'🌿 Species: {species_list}')

🌿 Species: ['Diya_Na', 'Gammiris', 'Ingini', 'Iriveriya', 'Kapparawalliya', 'Kora_Kaha', 'Kuringchan', 'Kurundu', 'Masbadda', 'Na', 'Rathu_Koboleela', 'Sudu_Koboleela', 'Wali_Kaha']


## Cell 3b — Threshold & Template Lookup

In [15]:
def get_thresholds(species=None, view=None):
    """Layered lookup: DEFAULT_THRESHOLDS <- species+view override."""
    t = dict(DEFAULT_THRESHOLDS)
    species_over = SPECIES_THRESHOLD_OVERRIDES.get(species, {})
    view_over = species_over.get(view, {}) if view else {}
    t.update(view_over)
    return t

In [16]:
def get_hull_baseline(species=None):
    return SPECIES_HULL_BASELINE.get(species, DEFAULT_HULL_RATIO_BASELINE)

In [17]:
def get_shape_template(species=None):
    """Decodes and returns the species' shape template as a float32 array
    in [0,1], or None if not calibrated (caller should fall back to the
    hull-ratio heuristic)."""
    b64 = SPECIES_SHAPE_TEMPLATES_B64.get(species)
    if b64 is None:
        return None
    png_bytes = base64.b64decode(b64)
    arr = np.frombuffer(png_bytes, dtype=np.uint8)
    template = cv2.imdecode(arr, cv2.IMREAD_GRAYSCALE)
    if template is None:
        return None
    return template.astype(np.float32) / 255.0

In [18]:
print('✅ Threshold/template lookups ready!')

✅ Threshold/template lookups ready!


## Cell 3c — Calibration: Per-View Colour + Pooled-View Shape Template

Run once per species. Colour thresholds are calibrated separately for
top and bottom (since the two sides look different); the shape template
pools BOTH views' healthy images (same physical outline either way).

In [19]:
def get_calibration_images(species, category, view):
    folder = os.path.join(HEALTH_CALIBRATION_PATH, species, category, view)
    return (
        glob(os.path.join(folder, '*.jpg'))  +
        glob(os.path.join(folder, '*.jpeg')) +
        glob(os.path.join(folder, '*.png'))
    )

In [20]:
def calibrate_colour_for_view(species, view):
    healthy_imgs = get_calibration_images(species, 'healthy', view)
    damaged_imgs = get_calibration_images(species, 'damaged', view)

    if not healthy_imgs:
        print(f'  ⚠️  No healthy/{view} reference images for {species}')
        return None

    healthy_h, healthy_s, healthy_v = [], [], []
    for path in healthy_imgs:
        img = cv2.imread(path)
        if img is None:
            continue
        img = resize_with_padding(img, WORK_SIZE)
        mask = segment_leaf(img)
        if cv2.countNonZero(mask) == 0:
            continue
        hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)
        leaf_bool = mask > 0
        healthy_h.append(hsv[:, :, 0][leaf_bool])
        healthy_s.append(hsv[:, :, 1][leaf_bool])
        healthy_v.append(hsv[:, :, 2][leaf_bool])

    healthy_h = np.concatenate(healthy_h)
    healthy_s = np.concatenate(healthy_s)
    healthy_v = np.concatenate(healthy_v)

    suggested = {
        'HEALTHY_HUE_RANGE': (int(np.percentile(healthy_h, 2)), int(np.percentile(healthy_h, 98))),
        'HEALTHY_MIN_SAT': max(0, int(np.percentile(healthy_s, 5)) - 5),
        'HEALTHY_MIN_VAL': max(0, int(np.percentile(healthy_v, 5)) - 5),
    }

    if damaged_imgs:
        damaged_v_dark, damaged_s_dark = [], []
        for path in damaged_imgs:
            img = cv2.imread(path)
            if img is None:
                continue
            img = resize_with_padding(img, WORK_SIZE)
            mask = segment_leaf(img)
            if cv2.countNonZero(mask) == 0:
                continue
            hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)
            leaf_bool = mask > 0
            v_vals = hsv[:, :, 2][leaf_bool]
            s_vals = hsv[:, :, 1][leaf_bool]
            dark_cutoff = np.percentile(v_vals, 30)
            dark_pixels = v_vals <= dark_cutoff
            damaged_v_dark.append(v_vals[dark_pixels])
            damaged_s_dark.append(s_vals[dark_pixels])
        if damaged_v_dark:
            damaged_v_dark = np.concatenate(damaged_v_dark)
            damaged_s_dark = np.concatenate(damaged_s_dark)
            suggested['BLACK_MAX_VAL'] = int(np.percentile(damaged_v_dark, 60))
            suggested['BLACK_MAX_SAT'] = int(np.percentile(damaged_s_dark, 60))

    print(f'  {view}: calibrated from {len(healthy_imgs)} healthy, {len(damaged_imgs)} damaged leaves')
    return suggested

In [21]:
def calibrate_species(species, show_plot=True):
    print(f'\n🌿 Calibrating {species}...')

    threshold_result = {}
    for view in VIEWS:
        result = calibrate_colour_for_view(species, view)
        if result is not None:
            threshold_result[view] = result

    print(f"\n   Paste into SPECIES_THRESHOLD_OVERRIDES (Cell 3):")
    print(f"   '{species}': {threshold_result!r},")

    # --- Shape template: pool healthy leaves from BOTH views ---
    shape_masks = []
    for view in VIEWS:
        for path in get_calibration_images(species, 'healthy', view):
            img = cv2.imread(path)
            if img is None:
                continue
            img = resize_with_padding(img, WORK_SIZE)
            mask = segment_leaf(img)
            if cv2.countNonZero(mask) > 0:
                shape_masks.append(mask)

    template, n_used = build_shape_template(shape_masks)
    if template is None:
        print(f'\n   ⚠️  Not enough usable leaves ({n_used}) to build a shape template '
              f'(need >= 3). Margin loss will fall back to the hull-ratio heuristic.')
        b64 = None
    else:
        b64 = template_to_base64_png(template)
        print(f'\n   Shape template built from {n_used} pooled top+bottom healthy leaves.')
        print(f"   Paste into SPECIES_SHAPE_TEMPLATES_B64 (Cell 3):")
        print(f"   '{species}': '{b64[:60]}...',   # (full string is {len(b64)} chars — copy from the")
        print(f"                                    #  variable below, this is truncated for display)")

        if show_plot:
            fig, ax = plt.subplots(figsize=(4, 4))
            ax.imshow(template, cmap='Greens')
            ax.set_title(f'{species} — Shape Template\n(brighter = more consistently leaf)', fontsize=9)
            ax.axis('off')
            plt.tight_layout()
            plt.show()

    return {'thresholds': threshold_result, 'template_b64': b64}

In [22]:
print('✅ calibrate_species() ready! e.g.: result = calibrate_species("Diya Na")')
print('   Full base64 string (if you need to copy it in full): result["template_b64"]')

✅ calibrate_species() ready! e.g.: result = calibrate_species("Diya Na")
   Full base64 string (if you need to copy it in full): result["template_b64"]


## Cell 4 — Helper Functions

In [ ]:
def bgr2rgb(img):
    return cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

In [ ]:
def get_images(base_path, species, view):
    return (
        glob(os.path.join(base_path, species, view, '*.jpg'))  +
        glob(os.path.join(base_path, species, view, '*.jpeg')) +
        glob(os.path.join(base_path, species, view, '*.png'))
    )

In [ ]:
def resize_with_padding(image, target_size, pad_value=255):
    h, w = image.shape[:2]
    target_w, target_h = target_size
    scale = min(target_w / w, target_h / h)
    new_w, new_h = int(round(w * scale)), int(round(h * scale))
    resized = cv2.resize(image, (new_w, new_h))
    canvas = np.full((target_h, target_w, 3), pad_value, dtype=np.uint8)
    x_off = (target_w - new_w) // 2
    y_off = (target_h - new_h) // 2
    canvas[y_off:y_off + new_h, x_off:x_off + new_w] = resized
    return canvas

In [ ]:
def extract_leaf_id(filename):
    match = re.search(r'\((\d+)\)', filename)
    if match is None:
        raise ValueError(f'Could not find a leaf ID "(n)" in filename: {filename}')
    return match.group(1)

In [ ]:
def group_by_leaf(image_paths):
    groups = {}
    for path in image_paths:
        leaf_id = extract_leaf_id(os.path.basename(path))
        groups.setdefault(leaf_id, []).append(path)
    return groups

In [ ]:
print('✅ Helper functions defined!')

## Cell 5 — Leaf Isolation (reused from 02_Preprocessing.ipynb)

In [ ]:
def get_border_connected_background(candidate_bg, bridge_break_px=7):
    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (bridge_break_px, bridge_break_px))
    eroded_bg = cv2.erode(candidate_bg, kernel)
    h, w = eroded_bg.shape
    padded = cv2.copyMakeBorder(eroded_bg, 1, 1, 1, 1, cv2.BORDER_CONSTANT, value=255)
    flood_seed = np.zeros((h + 4, w + 4), np.uint8)
    filled = padded.copy()
    cv2.floodFill(filled, flood_seed, (0, 0), 128)
    border_bg = ((filled == 128).astype(np.uint8) * 255)[1:-1, 1:-1]
    border_bg = cv2.dilate(border_bg, kernel)
    border_bg = cv2.bitwise_and(border_bg, candidate_bg)
    return border_bg

In [ ]:
def remove_small_blobs(mask, min_area=100):
    num_labels, labels, stats, _ = cv2.connectedComponentsWithStats(mask, connectivity=8)
    clean = np.zeros_like(mask)
    for label in range(1, num_labels):
        if stats[label, cv2.CC_STAT_AREA] >= min_area:
            clean[labels == label] = 255
    return clean

In [ ]:
def keep_largest_contours(mask, keep_ratio=0.05, min_absolute_area=1500):
    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if not contours:
        return mask
    areas = [cv2.contourArea(c) for c in contours]
    largest_area = max(areas)
    clean = np.zeros_like(mask)
    for c, a in zip(contours, areas):
        if a >= largest_area * keep_ratio or a >= min_absolute_area:
            cv2.drawContours(clean, [c], -1, 255, thickness=cv2.FILLED)
    return clean

In [ ]:
def segment_leaf(image, dark_v_thresh=90, min_shadow_area=200, max_artifact_area=150,
                  bridge_break_px=7, morph_kernel_size=3, min_blob_area=100,
                  otsu_relax_factor=0.8):
    hsv = cv2.cvtColor(image, cv2.COLOR_BGR2HSV)
    s = hsv[:, :, 1]
    v = hsv[:, :, 2]

    otsu_thresh, _ = cv2.threshold(s, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    relaxed_thresh = otsu_thresh * otsu_relax_factor
    _, candidate_leaf = cv2.threshold(s, relaxed_thresh, 255, cv2.THRESH_BINARY)
    candidate_bg = cv2.bitwise_not(candidate_leaf)

    border_bg = get_border_connected_background(candidate_bg, bridge_break_px)
    enclosed = cv2.bitwise_and(candidate_bg, cv2.bitwise_not(border_bg))

    num_labels, labels, stats, _ = cv2.connectedComponentsWithStats(enclosed, connectivity=8)
    dark_enclosed = np.zeros_like(enclosed)
    undecided = []

    for label in range(1, num_labels):
        comp = (labels == label)
        area = stats[label, cv2.CC_STAT_AREA]
        if area < min_shadow_area:
            undecided.append(label)
            continue
        if np.median(v[comp]) < dark_v_thresh:
            dark_enclosed[comp] = 255
        else:
            undecided.append(label)

    kernel_adj = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (11, 11))
    dilated_dark = cv2.dilate(dark_enclosed, kernel_adj, iterations=1)

    for label in undecided:
        comp = (labels == label)
        area = stats[label, cv2.CC_STAT_AREA]
        if area < max_artifact_area and np.any(dilated_dark[comp]):
            dark_enclosed[comp] = 255

    background_mask = cv2.bitwise_or(border_bg, dark_enclosed)
    leaf_mask = cv2.bitwise_not(background_mask)
    leaf_mask = remove_small_blobs(leaf_mask, min_area=min_blob_area)

    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (morph_kernel_size, morph_kernel_size))
    leaf_mask = cv2.morphologyEx(leaf_mask, cv2.MORPH_CLOSE, kernel, iterations=2)
    leaf_mask = cv2.morphologyEx(leaf_mask, cv2.MORPH_OPEN, kernel, iterations=1)

    leaf_mask = keep_largest_contours(leaf_mask)
    return leaf_mask

In [ ]:
print('✅ Leaf isolation (segment_leaf) ready!')

## Cell 5b — Interior Hole Detection (Insect/Chew Holes)

In [ ]:
def get_background_reference_color(image_bgr, sample_size=20):
    """Median BGR sampled from the ORIGINAL image's four corners -- must
    be called on the un-padded image (see evaluate_single_view_health),
    since resize_with_padding() fills padding with pure white, which is
    NOT your real photographed background."""
    h, w = image_bgr.shape[:2]
    patches = [
        image_bgr[0:sample_size, 0:sample_size],
        image_bgr[0:sample_size, w - sample_size:w],
        image_bgr[h - sample_size:h, 0:sample_size],
        image_bgr[h - sample_size:h, w - sample_size:w],
    ]
    all_px = np.concatenate([p.reshape(-1, 3) for p in patches], axis=0)
    return np.median(all_px, axis=0)

In [ ]:
def detect_holes(image_bgr, leaf_mask, bg_color,
                  color_tolerance=BG_COLOR_TOLERANCE,
                  bridge_break_px=HOLE_BRIDGE_BREAK_PX):
    diff = np.linalg.norm(
        image_bgr.astype(np.float32) - bg_color.astype(np.float32), axis=2
    )
    is_bg_color = (diff <= color_tolerance).astype(np.uint8) * 255
    true_exterior = get_border_connected_background(is_bg_color, bridge_break_px)
    holes = cv2.bitwise_and(is_bg_color, cv2.bitwise_not(true_exterior))
    holes = cv2.bitwise_and(holes, leaf_mask)
    holes = remove_small_blobs(holes, min_area=10)
    return holes

In [ ]:
print('✅ detect_holes() ready!')

## Cell 5c — Shape Alignment & Reference-Template Margin Loss

Aligns a leaf mask to a canonical pose (PCA rotation to vertical, tip
pointing up, rescaled to a common length) so leaves of different sizes/
orientations/photo crops become directly comparable. Used both to BUILD
a per-species template (Cell 3c, from calibration leaves) and to SCORE
a new leaf against it (Cell 8).

In [ ]:
def align_leaf_mask(mask, canvas_size=SHAPE_CANVAS_SIZE, target_length=SHAPE_TARGET_LENGTH):
    """
    Returns the mask warped into a canonical canvas_size x canvas_size
    pose: centred, major axis vertical, narrow (tip) end pointing up,
    rescaled so its length along the major axis = target_length.
    Returns None if the mask is empty/degenerate.
    """
    ys, xs = np.nonzero(mask)
    if len(xs) < 50:
        return None

    coords = np.stack([xs, ys], axis=1).astype(np.float64)
    mean = coords.mean(axis=0)
    centered = coords - mean
    cov = np.cov(centered.T)
    eigvals, eigvecs = np.linalg.eigh(cov)
    major = eigvecs[:, np.argmax(eigvals)]
    angle_deg = np.degrees(np.arctan2(major[1], major[0]))
    rotation_angle = 90 - angle_deg

    proj = centered @ major
    length = proj.max() - proj.min()
    if length <= 0:
        return None
    scale = target_length / length

    cx, cy = mean
    M = cv2.getRotationMatrix2D((cx, cy), rotation_angle, scale)
    M[0, 2] += canvas_size / 2 - cx
    M[1, 2] += canvas_size / 2 - cy
    aligned = cv2.warpAffine(mask, M, (canvas_size, canvas_size),
                              flags=cv2.INTER_NEAREST, borderValue=0)

    # Resolve the 180-degree ambiguity left by PCA: canonically, the
    # NARROWER end (leaf tip/apex) points up, regardless of how the leaf
    # happened to be placed in the original photo.
    ys2 = np.nonzero(aligned)[0]
    if len(ys2) == 0:
        return None
    y_min, y_max = ys2.min(), ys2.max()
    span = y_max - y_min
    if span >= 10:
        band = max(1, span // 10)
        top_width = np.count_nonzero(aligned[y_min:y_min + band, :].any(axis=0))
        bottom_width = np.count_nonzero(aligned[y_max - band:y_max + 1, :].any(axis=0))
        if bottom_width < top_width:
            aligned = cv2.rotate(aligned, cv2.ROTATE_180)

    return aligned

In [ ]:
def build_shape_template(masks, canvas_size=SHAPE_CANVAS_SIZE, target_length=SHAPE_TARGET_LENGTH):
    """
    Averages multiple aligned healthy-leaf masks into a probability
    template (float32, 0-1): for each canvas pixel, the fraction of
    reference leaves that had tissue there.
    """
    aligned_masks = []
    for mask in masks:
        aligned = align_leaf_mask(mask, canvas_size, target_length)
        if aligned is not None:
            aligned_masks.append((aligned > 0).astype(np.float32))

    if len(aligned_masks) < 3:
        return None, len(aligned_masks)

    template = np.mean(aligned_masks, axis=0)
    return template, len(aligned_masks)

In [ ]:
def template_to_base64_png(template_float):
    """Encodes a float32 [0,1] template as base64 PNG text, for pasting
    into SPECIES_SHAPE_TEMPLATES_B64."""
    img_u8 = (template_float * 255).astype(np.uint8)
    success, encoded = cv2.imencode('.png', img_u8)
    if not success:
        return None
    return base64.b64encode(encoded.tobytes()).decode('ascii')

In [ ]:
def score_against_template(leaf_mask, template, canvas_size=SHAPE_CANVAS_SIZE, target_length=SHAPE_TARGET_LENGTH):
    """
    Returns margin_loss_pct: how much of the template's expected leaf
    area is missing from this leaf, weighted by how reliably each
    canvas location is covered in healthy reference leaves (so naturally
    variable spots near lobes/notches don't get penalised as heavily as
    the reliably-solid main lamina).
    """
    aligned = align_leaf_mask(leaf_mask, canvas_size, target_length)
    if aligned is None:
        return None

    present = (aligned > 0).astype(np.float32)
    expected_mass = template.sum()
    if expected_mass == 0:
        return 0.0

    present_and_expected = (template * present).sum()
    coverage_ratio = present_and_expected / expected_mass
    loss_pct = max(0.0, (1 - coverage_ratio) * 100)
    return round(min(loss_pct, 100.0), 2)

In [ ]:
print('✅ Shape alignment & template scoring ready!')

## Cell 5d — Margin-Loss: Convex-Hull Fallback (for un-calibrated species)

In [ ]:
def compute_hull_ratio(leaf_mask):
    contours, _ = cv2.findContours(leaf_mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if not contours:
        return None, 0, 0
    largest = max(contours, key=cv2.contourArea)
    leaf_area = cv2.contourArea(largest)
    hull = cv2.convexHull(largest)
    hull_area = cv2.contourArea(hull)
    if hull_area == 0:
        return None, leaf_area, 0
    return leaf_area / hull_area, leaf_area, hull_area

In [ ]:
def estimate_margin_loss_pct(leaf_mask, species=None):
    """Uses the species' shape template if one has been calibrated;
    otherwise falls back to the convex-hull-ratio heuristic."""
    template = get_shape_template(species)
    if template is not None:
        score = score_against_template(leaf_mask, template)
        if score is not None:
            return score

    ratio, leaf_area, hull_area = compute_hull_ratio(leaf_mask)
    if ratio is None:
        return 0.0
    baseline = get_hull_baseline(species)
    loss_pct = max(0.0, (baseline - ratio) / baseline * 100)
    return round(min(loss_pct, 100.0), 2)

In [ ]:
print('✅ Margin-loss estimation ready (template-based, hull fallback)!')

## Cell 6 — Colour Classification Rules (run on TISSUE only, holes excluded)

In [ ]:
def build_health_masks(image_bgr, tissue_mask, thresholds):
    hsv = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2HSV)
    h, s, v = hsv[:, :, 0], hsv[:, :, 1], hsv[:, :, 2]
    tissue_bool = tissue_mask > 0

    def to_mask(bool_arr):
        return (bool_arr & tissue_bool).astype(np.uint8) * 255

    hh = thresholds['HEALTHY_HUE_RANGE']
    yh = thresholds['YELLOW_HUE_RANGE']
    bh = thresholds['BROWN_HUE_RANGE']

    healthy = to_mask((h >= hh[0]) & (h <= hh[1]) &
                       (s >= thresholds['HEALTHY_MIN_SAT']) &
                       (v >= thresholds['HEALTHY_MIN_VAL']))
    yellow = to_mask((h >= yh[0]) & (h <= yh[1]) & (s >= thresholds['YELLOW_MIN_SAT']))
    brown = to_mask((h >= bh[0]) & (h <= bh[1]) & (v <= thresholds['BROWN_MAX_VAL']))
    black = to_mask((v <= thresholds['BLACK_MAX_VAL']) & (s <= thresholds['BLACK_MAX_SAT']))
    pale = to_mask((s <= thresholds['PALE_MAX_SAT']) & (v >= thresholds['PALE_MIN_VAL']))

    yellow = cv2.bitwise_and(yellow, cv2.bitwise_not(healthy))
    brown  = cv2.bitwise_and(brown,  cv2.bitwise_not(healthy))
    black  = cv2.bitwise_and(black,  cv2.bitwise_not(healthy))
    pale   = cv2.bitwise_and(pale,   cv2.bitwise_not(healthy))

    brown  = cv2.bitwise_and(brown,  cv2.bitwise_not(black))
    pale   = cv2.bitwise_and(pale,   cv2.bitwise_not(black))
    yellow = cv2.bitwise_and(yellow, cv2.bitwise_not(black))

    pale   = cv2.bitwise_and(pale,   cv2.bitwise_not(brown))
    yellow = cv2.bitwise_and(yellow, cv2.bitwise_not(brown))

    yellow = cv2.bitwise_and(yellow, cv2.bitwise_not(pale))

    return {'healthy': healthy, 'yellow': yellow, 'brown': brown, 'black': black, 'pale': pale}

In [ ]:
print('✅ Health classification rules defined!')

## Cell 7 — Noise Cleanup for Damage Masks

In [ ]:
def clean_health_masks(masks, min_area=MIN_DAMAGE_BLOB_AREA):
    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (3, 3))
    cleaned = {}
    for name, mask in masks.items():
        opened = cv2.morphologyEx(mask, cv2.MORPH_OPEN, kernel, iterations=1)
        cleaned[name] = remove_small_blobs(opened, min_area=min_area)
    return cleaned

In [ ]:
print('✅ Noise cleanup ready!')

## Cell 8 — Full Health Pipeline (Colour + Structural, view-aware)

In [ ]:
def evaluate_single_view_health(image_bgr, species=None, view=None):
    """
    view: 'top' or 'bottom' -- selects the correct per-view colour
    thresholds. Shape template/hull baseline is NOT view-dependent.
    """
    thresholds = get_thresholds(species, view)

    # Sample background BEFORE padding (padding is pure white, not your
    # real photographed background).
    bg_color = get_background_reference_color(image_bgr)

    img = resize_with_padding(image_bgr, WORK_SIZE)
    leaf_mask = segment_leaf(img)

    leaf_silhouette_px = int(cv2.countNonZero(leaf_mask))
    if leaf_silhouette_px == 0:
        return None

    holes_mask = detect_holes(img, leaf_mask, bg_color)
    hole_px = int(cv2.countNonZero(holes_mask))

    tissue_mask = cv2.bitwise_and(leaf_mask, cv2.bitwise_not(holes_mask))
    tissue_px = int(cv2.countNonZero(tissue_mask))
    if tissue_px == 0:
        return None

    masks = build_health_masks(img, tissue_mask, thresholds)
    masks = clean_health_masks(masks)
    counts = {name: int(cv2.countNonZero(m)) for name, m in masks.items()}
    damage_px = counts['yellow'] + counts['brown'] + counts['black'] + counts['pale']
    unclassified_px = max(0, tissue_px - counts['healthy'] - damage_px)

    tissue_quality_pct = round(counts['healthy'] / tissue_px * 100, 2)
    hole_pct = round(hole_px / leaf_silhouette_px * 100, 2)
    margin_loss_pct = estimate_margin_loss_pct(leaf_mask, species=species)

    final_health_pct = round(
        tissue_quality_pct * (1 - hole_pct / 100) * (1 - margin_loss_pct / 100), 2
    )

    return {
        'leaf_area_px': leaf_silhouette_px,
        'tissue_quality_pct': tissue_quality_pct,
        'yellow_pct': round(counts['yellow'] / tissue_px * 100, 2),
        'brown_pct': round(counts['brown'] / tissue_px * 100, 2),
        'black_pct': round(counts['black'] / tissue_px * 100, 2),
        'pale_pct': round(counts['pale'] / tissue_px * 100, 2),
        'unclassified_pct': round(unclassified_px / tissue_px * 100, 2),
        'hole_pct': hole_pct,
        'margin_loss_pct': margin_loss_pct,
        'final_health_pct': final_health_pct,
        'masks': masks,
        'holes_mask': holes_mask,
        'leaf_mask': leaf_mask,
        'working_image': img,
    }

In [ ]:
print('✅ evaluate_single_view_health() ready (view-aware)!')

## Cell 9 — Test on ONE Image, See Every Step, and Verify Template Alignment

In [ ]:
sample_species = species_list[0] if species_list else None

In [ ]:
if sample_species is None:
    print('⚠️  No species folders found under HEALTH_RAW_PATH yet — add data first.')
else:
    sample_view = VIEWS[0]
    sample_imgs = get_images(HEALTH_RAW_PATH, sample_species, sample_view)

    if not sample_imgs:
        print(f'⚠️  No images found in {HEALTH_RAW_PATH}/{sample_species}/{sample_view}/')
    else:
        sample_bgr = cv2.imread(random.choice(sample_imgs))
        report = evaluate_single_view_health(sample_bgr, species=sample_species, view=sample_view)

        if report is None:
            print('⚠️  Leaf mask came back empty for this image — check segmentation params.')
        else:
            overlay = report['working_image'].copy()
            overlay[report['masks']['healthy'] > 0] = (0, 200, 0)
            overlay[report['masks']['yellow'] > 0]  = (0, 220, 220)
            overlay[report['masks']['brown'] > 0]   = (19, 69, 139)
            overlay[report['masks']['black'] > 0]   = (0, 0, 0)
            overlay[report['masks']['pale'] > 0]    = (200, 200, 255)
            overlay[report['holes_mask'] > 0]       = (255, 0, 255)

            # NEW: visualize the aligned mask next to the template (if
            # calibrated) so you can SEE whether alignment/scoring makes
            # sense, not just trust the number blindly.
            template = get_shape_template(sample_species)
            aligned = align_leaf_mask(report['leaf_mask'])

            steps = [(bgr2rgb(report['working_image']), 'Resized 512×512'),
                     (bgr2rgb(overlay), 'Colour + Hole Overlay (magenta=hole)')]

            if template is not None and aligned is not None:
                compare = np.zeros((*template.shape, 3), dtype=np.uint8)
                compare[:, :, 1] = (template * 255).astype(np.uint8)   # green = template (expected)
                compare[:, :, 2] = aligned                              # red = this leaf (actual)
                steps.append((compare, 'Template (green) vs This Leaf (red) — overlap=yellow'))
            else:
                steps.append((aligned if aligned is not None else np.zeros((400, 400), dtype=np.uint8),
                               'Aligned shape (no template calibrated yet)'))

            fig, axes = plt.subplots(1, len(steps), figsize=(16, 5))
            fig.suptitle(
                f"🌿 {sample_species} ({sample_view}) — Final Health: {report['final_health_pct']}% "
                f"(quality {report['tissue_quality_pct']}%, holes {report['hole_pct']}%, "
                f"margin loss {report['margin_loss_pct']}%)",
                fontsize=11, fontweight='bold'
            )
            for ax, (img_step, title) in zip(axes, steps):
                ax.imshow(img_step, cmap='gray' if len(img_step.shape) == 2 else None)
                ax.set_title(title, fontsize=9, fontweight='bold')
                ax.axis('off')
            plt.tight_layout()
            plt.savefig(f'{LOGS_PATH}/03_health_eval_steps.png', bbox_inches='tight', dpi=120)
            plt.show()

            print(f"  Tissue quality (colour)   : {report['tissue_quality_pct']}%")
            print(f"  Hole area                 : {report['hole_pct']}%")
            print(f"  Margin loss               : {report['margin_loss_pct']}% "
                  f"({'template-based' if template is not None else 'hull-ratio fallback'})")
            print(f"  ➜ FINAL HEALTH            : {report['final_health_pct']}%")
            print()
            print('👉 If the template/leaf overlay looks badly misaligned (rotated wrong,')
            print('   mirrored), exclude that calibration image and re-run calibrate_species().')

## Cell 10 — Combine Top + Bottom Views (view-aware thresholds, area-weighted FINAL health)

In [ ]:
def evaluate_full_leaf_health(top_bgr, bottom_bgr, species=None):
    top_report = evaluate_single_view_health(top_bgr, species=species, view='top') if top_bgr is not None else None
    bottom_report = evaluate_single_view_health(bottom_bgr, species=species, view='bottom') if bottom_bgr is not None else None

    if top_report is None and bottom_report is None:
        return None

    if top_report is None:
        combined_pct = bottom_report['final_health_pct']
    elif bottom_report is None:
        combined_pct = top_report['final_health_pct']
    else:
        top_area = top_report['leaf_area_px']
        bottom_area = bottom_report['leaf_area_px']
        combined_pct = round(
            (top_report['final_health_pct'] * top_area +
             bottom_report['final_health_pct'] * bottom_area) /
            (top_area + bottom_area),
            2
        )

    return {
        'top_final_health_pct': top_report['final_health_pct'] if top_report else None,
        'bottom_final_health_pct': bottom_report['final_health_pct'] if bottom_report else None,
        'top_leaf_area_px': top_report['leaf_area_px'] if top_report else None,
        'bottom_leaf_area_px': bottom_report['leaf_area_px'] if bottom_report else None,
        'combined_final_health_pct': combined_pct,
    }

In [ ]:
print('✅ evaluate_full_leaf_health() ready!')

## Cell 3c — Calibration: Per-View Colour + Pooled-View Shape Template

Run once per species. Colour thresholds are calibrated separately for
top and bottom (since the two sides look different); the shape template
pools BOTH views' healthy images (same physical outline either way).

In [ ]:
def get_calibration_images(species, category, view):
    folder = os.path.join(HEALTH_CALIBRATION_PATH, species, category, view)
    return (
        glob(os.path.join(folder, '*.jpg'))  +
        glob(os.path.join(folder, '*.jpeg')) +
        glob(os.path.join(folder, '*.png'))
    )

In [ ]:
def calibrate_colour_for_view(species, view):
    healthy_imgs = get_calibration_images(species, 'healthy', view)
    damaged_imgs = get_calibration_images(species, 'damaged', view)

    if not healthy_imgs:
        print(f'  ⚠️  No healthy/{view} reference images for {species}')
        return None

    healthy_h, healthy_s, healthy_v = [], [], []
    for path in healthy_imgs:
        img = cv2.imread(path)
        if img is None:
            continue
        img = resize_with_padding(img, WORK_SIZE)
        mask = segment_leaf(img)
        if cv2.countNonZero(mask) == 0:
            continue
        hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)
        leaf_bool = mask > 0
        healthy_h.append(hsv[:, :, 0][leaf_bool])
        healthy_s.append(hsv[:, :, 1][leaf_bool])
        healthy_v.append(hsv[:, :, 2][leaf_bool])

    healthy_h = np.concatenate(healthy_h)
    healthy_s = np.concatenate(healthy_s)
    healthy_v = np.concatenate(healthy_v)

    suggested = {
        'HEALTHY_HUE_RANGE': (int(np.percentile(healthy_h, 2)), int(np.percentile(healthy_h, 98))),
        'HEALTHY_MIN_SAT': max(0, int(np.percentile(healthy_s, 5)) - 5),
        'HEALTHY_MIN_VAL': max(0, int(np.percentile(healthy_v, 5)) - 5),
    }

    if damaged_imgs:
        damaged_v_dark, damaged_s_dark = [], []
        for path in damaged_imgs:
            img = cv2.imread(path)
            if img is None:
                continue
            img = resize_with_padding(img, WORK_SIZE)
            mask = segment_leaf(img)
            if cv2.countNonZero(mask) == 0:
                continue
            hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)
            leaf_bool = mask > 0
            v_vals = hsv[:, :, 2][leaf_bool]
            s_vals = hsv[:, :, 1][leaf_bool]
            dark_cutoff = np.percentile(v_vals, 30)
            dark_pixels = v_vals <= dark_cutoff
            damaged_v_dark.append(v_vals[dark_pixels])
            damaged_s_dark.append(s_vals[dark_pixels])
        if damaged_v_dark:
            damaged_v_dark = np.concatenate(damaged_v_dark)
            damaged_s_dark = np.concatenate(damaged_s_dark)
            suggested['BLACK_MAX_VAL'] = int(np.percentile(damaged_v_dark, 60))
            suggested['BLACK_MAX_SAT'] = int(np.percentile(damaged_s_dark, 60))

    print(f'  {view}: calibrated from {len(healthy_imgs)} healthy, {len(damaged_imgs)} damaged leaves')
    return suggested

In [ ]:
def calibrate_species(species, show_plot=True):
    print(f'\n🌿 Calibrating {species}...')

    threshold_result = {}
    for view in VIEWS:
        result = calibrate_colour_for_view(species, view)
        if result is not None:
            threshold_result[view] = result

    print(f"\n   Paste into SPECIES_THRESHOLD_OVERRIDES (Cell 3):")
    print(f"   '{species}': {threshold_result!r},")

    # --- Shape template: pool healthy leaves from BOTH views ---
    shape_masks = []
    for view in VIEWS:
        for path in get_calibration_images(species, 'healthy', view):
            img = cv2.imread(path)
            if img is None:
                continue
            img = resize_with_padding(img, WORK_SIZE)
            mask = segment_leaf(img)
            if cv2.countNonZero(mask) > 0:
                shape_masks.append(mask)

    template, n_used = build_shape_template(shape_masks)
    if template is None:
        print(f'\n   ⚠️  Not enough usable leaves ({n_used}) to build a shape template '
              f'(need >= 3). Margin loss will fall back to the hull-ratio heuristic.')
        b64 = None
    else:
        b64 = template_to_base64_png(template)
        print(f'\n   Shape template built from {n_used} pooled top+bottom healthy leaves.')
        print(f"   Paste into SPECIES_SHAPE_TEMPLATES_B64 (Cell 3):")
        print(f"   '{species}': '{b64[:60]}...',   # (full string is {len(b64)} chars — copy from the")
        print(f"                                    #  variable below, this is truncated for display)")

        if show_plot:
            fig, ax = plt.subplots(figsize=(4, 4))
            ax.imshow(template, cmap='Greens')
            ax.set_title(f'{species} — Shape Template\n(brighter = more consistently leaf)', fontsize=9)
            ax.axis('off')
            plt.tight_layout()
            plt.show()

    return {'thresholds': threshold_result, 'template_b64': b64}

In [ ]:
print('✅ calibrate_species() ready! e.g.: result = calibrate_species("Diya Na")')
print('   Full base64 string (if you need to copy it in full): result["template_b64"]')

## Cell 11 — Quality Control

In [ ]:
print('🔍 QUALITY CONTROL — Sampling 5 physical leaves per species')
print('=' * 62)

In [ ]:
qc_rows = []
for species in species_list:
    leaf_ids_per_view = {}
    for view in VIEWS:
        imgs = get_images(HEALTH_RAW_PATH, species, view)
        leaf_ids_per_view[view] = group_by_leaf(imgs)

    common_ids = sorted(set(leaf_ids_per_view['top']) & set(leaf_ids_per_view['bottom']))
    if not common_ids:
        print(f'  ⚠️  No paired top+bottom leaves found: {species}')
        continue

    for leaf_id in common_ids[:5]:
        top_path = leaf_ids_per_view['top'][leaf_id][0]
        bottom_path = leaf_ids_per_view['bottom'][leaf_id][0]
        top_img = cv2.imread(top_path)
        bottom_img = cv2.imread(bottom_path)

        result = evaluate_full_leaf_health(top_img, bottom_img, species=species)
        if result is None:
            qc_rows.append({'species': species, 'leaf_id': leaf_id, 'status': 'FAIL'})
            continue

        qc_rows.append({
            'species': species, 'leaf_id': leaf_id, 'status': 'PASS',
            'top_final_pct': result['top_final_health_pct'],
            'bottom_final_pct': result['bottom_final_health_pct'],
            'combined_final_pct': result['combined_final_health_pct'],
            'colour_calibrated': species in SPECIES_THRESHOLD_OVERRIDES,
            'shape_calibrated': species in SPECIES_SHAPE_TEMPLATES_B64,
        })

In [ ]:
qc_df = pd.DataFrame(qc_rows)
if len(qc_df):
    pass_rate = (qc_df.status == 'PASS').mean() * 100
    print(f'\n  Total sampled : {len(qc_df)}')
    print(f'  Pass rate     : {pass_rate:.1f}%')
    print(qc_df.to_string(index=False))
else:
    print('  ⚠️  No data to QC yet.')

## Cell 12 — Run on Full Dataset -> CSV

In [ ]:
print('🔄 Starting health evaluation on full dataset...')
print('=' * 65)

In [ ]:
results = []

In [ ]:
for species in species_list:
    leaf_ids_per_view = {}
    for view in VIEWS:
        imgs = get_images(HEALTH_RAW_PATH, species, view)
        leaf_ids_per_view[view] = group_by_leaf(imgs)

    common_ids = sorted(set(leaf_ids_per_view['top']) & set(leaf_ids_per_view['bottom']))

    for leaf_id in tqdm(common_ids, desc=f'  {species}'):
        top_path = leaf_ids_per_view['top'][leaf_id][0]
        bottom_path = leaf_ids_per_view['bottom'][leaf_id][0]

        top_img = cv2.imread(top_path)
        bottom_img = cv2.imread(bottom_path)

        result = evaluate_full_leaf_health(top_img, bottom_img, species=species)
        if result is None:
            continue

        results.append({
            'species': species,
            'leaf_id': leaf_id,
            'top_final_pct': result['top_final_health_pct'],
            'bottom_final_pct': result['bottom_final_health_pct'],
            'combined_final_pct': result['combined_final_health_pct'],
        })

        del top_img, bottom_img
    gc.collect()

In [ ]:
results_df = pd.DataFrame(results)
csv_path = os.path.join(HEALTH_RESULTS_PATH, 'leaf_health_report.csv')
results_df.to_csv(csv_path, index=False)

In [ ]:
print('=' * 65)
print(f'✅ Done! {len(results_df)} physical leaves evaluated.')
print(f'📄 Saved: {csv_path}')

## Cell 13 — Preview Results

In [ ]:
if len(results_df):
    print('📊 Average combined FINAL health % by species:')
    print(results_df.groupby('species')['combined_final_pct'].mean().round(1).to_string())
else:
    print('⚠️  No results to preview yet.')

## Cell 14 — Export Standalone Rule Module (No .pkl needed!)

In [ ]:
health_rules_source = '''# health_rules.py
"""
VedaVision Module 2 -- Health Evaluation (v4: colour + structural damage,
per-view thresholds, shape-template margin loss). Deterministic rule
pipeline -- no .pkl, this module IS the logic.

Import evaluate_full_leaf_health(top_bgr, bottom_bgr, species="Diya Na")
from your FastAPI route.
"""

import base64
import cv2
import numpy as np

WORK_SIZE = (512, 512)

DEFAULT_THRESHOLDS = ''' + repr(DEFAULT_THRESHOLDS) + '''
MIN_DAMAGE_BLOB_AREA = ''' + repr(MIN_DAMAGE_BLOB_AREA) + '''
SPECIES_THRESHOLD_OVERRIDES = ''' + repr(SPECIES_THRESHOLD_OVERRIDES) + '''

BG_COLOR_TOLERANCE = ''' + repr(BG_COLOR_TOLERANCE) + '''
HOLE_BRIDGE_BREAK_PX = ''' + repr(HOLE_BRIDGE_BREAK_PX) + '''

SHAPE_CANVAS_SIZE = ''' + repr(SHAPE_CANVAS_SIZE) + '''
SHAPE_TARGET_LENGTH = ''' + repr(SHAPE_TARGET_LENGTH) + '''

DEFAULT_HULL_RATIO_BASELINE = ''' + repr(DEFAULT_HULL_RATIO_BASELINE) + '''
SPECIES_HULL_BASELINE = ''' + repr(SPECIES_HULL_BASELINE) + '''
SPECIES_SHAPE_TEMPLATES_B64 = ''' + repr(SPECIES_SHAPE_TEMPLATES_B64) + '''


def get_thresholds(species=None, view=None):
    t = dict(DEFAULT_THRESHOLDS)
    species_over = SPECIES_THRESHOLD_OVERRIDES.get(species, {})
    view_over = species_over.get(view, {}) if view else {}
    t.update(view_over)
    return t


def get_hull_baseline(species=None):
    return SPECIES_HULL_BASELINE.get(species, DEFAULT_HULL_RATIO_BASELINE)


def get_shape_template(species=None):
    b64 = SPECIES_SHAPE_TEMPLATES_B64.get(species)
    if b64 is None:
        return None
    png_bytes = base64.b64decode(b64)
    arr = np.frombuffer(png_bytes, dtype=np.uint8)
    template = cv2.imdecode(arr, cv2.IMREAD_GRAYSCALE)
    if template is None:
        return None
    return template.astype(np.float32) / 255.0


def resize_with_padding(image, target_size, pad_value=255):
    h, w = image.shape[:2]
    target_w, target_h = target_size
    scale = min(target_w / w, target_h / h)
    new_w, new_h = int(round(w * scale)), int(round(h * scale))
    resized = cv2.resize(image, (new_w, new_h))
    canvas = np.full((target_h, target_w, 3), pad_value, dtype=np.uint8)
    x_off = (target_w - new_w) // 2
    y_off = (target_h - new_h) // 2
    canvas[y_off:y_off + new_h, x_off:x_off + new_w] = resized
    return canvas


def get_border_connected_background(candidate_bg, bridge_break_px=7):
    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (bridge_break_px, bridge_break_px))
    eroded_bg = cv2.erode(candidate_bg, kernel)
    h, w = eroded_bg.shape
    padded = cv2.copyMakeBorder(eroded_bg, 1, 1, 1, 1, cv2.BORDER_CONSTANT, value=255)
    flood_seed = np.zeros((h + 4, w + 4), np.uint8)
    filled = padded.copy()
    cv2.floodFill(filled, flood_seed, (0, 0), 128)
    border_bg = ((filled == 128).astype(np.uint8) * 255)[1:-1, 1:-1]
    border_bg = cv2.dilate(border_bg, kernel)
    border_bg = cv2.bitwise_and(border_bg, candidate_bg)
    return border_bg


def remove_small_blobs(mask, min_area=100):
    num_labels, labels, stats, _ = cv2.connectedComponentsWithStats(mask, connectivity=8)
    clean = np.zeros_like(mask)
    for label in range(1, num_labels):
        if stats[label, cv2.CC_STAT_AREA] >= min_area:
            clean[labels == label] = 255
    return clean


def keep_largest_contours(mask, keep_ratio=0.05, min_absolute_area=1500):
    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if not contours:
        return mask
    areas = [cv2.contourArea(c) for c in contours]
    largest_area = max(areas)
    clean = np.zeros_like(mask)
    for c, a in zip(contours, areas):
        if a >= largest_area * keep_ratio or a >= min_absolute_area:
            cv2.drawContours(clean, [c], -1, 255, thickness=cv2.FILLED)
    return clean


def segment_leaf(image, dark_v_thresh=90, min_shadow_area=200, max_artifact_area=150,
                  bridge_break_px=7, morph_kernel_size=3, min_blob_area=100,
                  otsu_relax_factor=0.8):
    hsv = cv2.cvtColor(image, cv2.COLOR_BGR2HSV)
    s = hsv[:, :, 1]
    v = hsv[:, :, 2]

    otsu_thresh, _ = cv2.threshold(s, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    relaxed_thresh = otsu_thresh * otsu_relax_factor
    _, candidate_leaf = cv2.threshold(s, relaxed_thresh, 255, cv2.THRESH_BINARY)
    candidate_bg = cv2.bitwise_not(candidate_leaf)

    border_bg = get_border_connected_background(candidate_bg, bridge_break_px)
    enclosed = cv2.bitwise_and(candidate_bg, cv2.bitwise_not(border_bg))

    num_labels, labels, stats, _ = cv2.connectedComponentsWithStats(enclosed, connectivity=8)
    dark_enclosed = np.zeros_like(enclosed)
    undecided = []

    for label in range(1, num_labels):
        comp = (labels == label)
        area = stats[label, cv2.CC_STAT_AREA]
        if area < min_shadow_area:
            undecided.append(label)
            continue
        if np.median(v[comp]) < dark_v_thresh:
            dark_enclosed[comp] = 255
        else:
            undecided.append(label)

    kernel_adj = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (11, 11))
    dilated_dark = cv2.dilate(dark_enclosed, kernel_adj, iterations=1)

    for label in undecided:
        comp = (labels == label)
        area = stats[label, cv2.CC_STAT_AREA]
        if area < max_artifact_area and np.any(dilated_dark[comp]):
            dark_enclosed[comp] = 255

    background_mask = cv2.bitwise_or(border_bg, dark_enclosed)
    leaf_mask = cv2.bitwise_not(background_mask)
    leaf_mask = remove_small_blobs(leaf_mask, min_area=min_blob_area)

    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (morph_kernel_size, morph_kernel_size))
    leaf_mask = cv2.morphologyEx(leaf_mask, cv2.MORPH_CLOSE, kernel, iterations=2)
    leaf_mask = cv2.morphologyEx(leaf_mask, cv2.MORPH_OPEN, kernel, iterations=1)

    leaf_mask = keep_largest_contours(leaf_mask)
    return leaf_mask


def get_background_reference_color(image_bgr, sample_size=20):
    h, w = image_bgr.shape[:2]
    patches = [
        image_bgr[0:sample_size, 0:sample_size],
        image_bgr[0:sample_size, w - sample_size:w],
        image_bgr[h - sample_size:h, 0:sample_size],
        image_bgr[h - sample_size:h, w - sample_size:w],
    ]
    all_px = np.concatenate([p.reshape(-1, 3) for p in patches], axis=0)
    return np.median(all_px, axis=0)


def detect_holes(image_bgr, leaf_mask, bg_color,
                  color_tolerance=BG_COLOR_TOLERANCE,
                  bridge_break_px=HOLE_BRIDGE_BREAK_PX):
    diff = np.linalg.norm(image_bgr.astype(np.float32) - bg_color.astype(np.float32), axis=2)
    is_bg_color = (diff <= color_tolerance).astype(np.uint8) * 255
    true_exterior = get_border_connected_background(is_bg_color, bridge_break_px)
    holes = cv2.bitwise_and(is_bg_color, cv2.bitwise_not(true_exterior))
    holes = cv2.bitwise_and(holes, leaf_mask)
    holes = remove_small_blobs(holes, min_area=10)
    return holes


def align_leaf_mask(mask, canvas_size=SHAPE_CANVAS_SIZE, target_length=SHAPE_TARGET_LENGTH):
    ys, xs = np.nonzero(mask)
    if len(xs) < 50:
        return None

    coords = np.stack([xs, ys], axis=1).astype(np.float64)
    mean = coords.mean(axis=0)
    centered = coords - mean
    cov = np.cov(centered.T)
    eigvals, eigvecs = np.linalg.eigh(cov)
    major = eigvecs[:, np.argmax(eigvals)]
    angle_deg = np.degrees(np.arctan2(major[1], major[0]))
    rotation_angle = 90 - angle_deg

    proj = centered @ major
    length = proj.max() - proj.min()
    if length <= 0:
        return None
    scale = target_length / length

    cx, cy = mean
    M = cv2.getRotationMatrix2D((cx, cy), rotation_angle, scale)
    M[0, 2] += canvas_size / 2 - cx
    M[1, 2] += canvas_size / 2 - cy
    aligned = cv2.warpAffine(mask, M, (canvas_size, canvas_size),
                              flags=cv2.INTER_NEAREST, borderValue=0)

    ys2 = np.nonzero(aligned)[0]
    if len(ys2) == 0:
        return None
    y_min, y_max = ys2.min(), ys2.max()
    span = y_max - y_min
    if span >= 10:
        band = max(1, span // 10)
        top_width = np.count_nonzero(aligned[y_min:y_min + band, :].any(axis=0))
        bottom_width = np.count_nonzero(aligned[y_max - band:y_max + 1, :].any(axis=0))
        if bottom_width < top_width:
            aligned = cv2.rotate(aligned, cv2.ROTATE_180)

    return aligned


def score_against_template(leaf_mask, template, canvas_size=SHAPE_CANVAS_SIZE, target_length=SHAPE_TARGET_LENGTH):
    aligned = align_leaf_mask(leaf_mask, canvas_size, target_length)
    if aligned is None:
        return None
    present = (aligned > 0).astype(np.float32)
    expected_mass = template.sum()
    if expected_mass == 0:
        return 0.0
    present_and_expected = (template * present).sum()
    coverage_ratio = present_and_expected / expected_mass
    loss_pct = max(0.0, (1 - coverage_ratio) * 100)
    return round(min(loss_pct, 100.0), 2)


def compute_hull_ratio(leaf_mask):
    contours, _ = cv2.findContours(leaf_mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if not contours:
        return None, 0, 0
    largest = max(contours, key=cv2.contourArea)
    leaf_area = cv2.contourArea(largest)
    hull = cv2.convexHull(largest)
    hull_area = cv2.contourArea(hull)
    if hull_area == 0:
        return None, leaf_area, 0
    return leaf_area / hull_area, leaf_area, hull_area


def estimate_margin_loss_pct(leaf_mask, species=None):
    template = get_shape_template(species)
    if template is not None:
        score = score_against_template(leaf_mask, template)
        if score is not None:
            return score
    ratio, leaf_area, hull_area = compute_hull_ratio(leaf_mask)
    if ratio is None:
        return 0.0
    baseline = get_hull_baseline(species)
    loss_pct = max(0.0, (baseline - ratio) / baseline * 100)
    return round(min(loss_pct, 100.0), 2)


def build_health_masks(image_bgr, tissue_mask, thresholds):
    hsv = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2HSV)
    h, s, v = hsv[:, :, 0], hsv[:, :, 1], hsv[:, :, 2]
    tissue_bool = tissue_mask > 0

    def to_mask(bool_arr):
        return (bool_arr & tissue_bool).astype(np.uint8) * 255

    hh = thresholds["HEALTHY_HUE_RANGE"]
    yh = thresholds["YELLOW_HUE_RANGE"]
    bh = thresholds["BROWN_HUE_RANGE"]

    healthy = to_mask((h >= hh[0]) & (h <= hh[1]) &
                       (s >= thresholds["HEALTHY_MIN_SAT"]) &
                       (v >= thresholds["HEALTHY_MIN_VAL"]))
    yellow = to_mask((h >= yh[0]) & (h <= yh[1]) & (s >= thresholds["YELLOW_MIN_SAT"]))
    brown = to_mask((h >= bh[0]) & (h <= bh[1]) & (v <= thresholds["BROWN_MAX_VAL"]))
    black = to_mask((v <= thresholds["BLACK_MAX_VAL"]) & (s <= thresholds["BLACK_MAX_SAT"]))
    pale = to_mask((s <= thresholds["PALE_MAX_SAT"]) & (v >= thresholds["PALE_MIN_VAL"]))

    yellow = cv2.bitwise_and(yellow, cv2.bitwise_not(healthy))
    brown = cv2.bitwise_and(brown, cv2.bitwise_not(healthy))
    black = cv2.bitwise_and(black, cv2.bitwise_not(healthy))
    pale = cv2.bitwise_and(pale, cv2.bitwise_not(healthy))

    brown = cv2.bitwise_and(brown, cv2.bitwise_not(black))
    pale = cv2.bitwise_and(pale, cv2.bitwise_not(black))
    yellow = cv2.bitwise_and(yellow, cv2.bitwise_not(black))

    pale = cv2.bitwise_and(pale, cv2.bitwise_not(brown))
    yellow = cv2.bitwise_and(yellow, cv2.bitwise_not(brown))

    yellow = cv2.bitwise_and(yellow, cv2.bitwise_not(pale))

    return {"healthy": healthy, "yellow": yellow, "brown": brown, "black": black, "pale": pale}


def clean_health_masks(masks, min_area=MIN_DAMAGE_BLOB_AREA):
    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (3, 3))
    cleaned = {}
    for name, mask in masks.items():
        opened = cv2.morphologyEx(mask, cv2.MORPH_OPEN, kernel, iterations=1)
        cleaned[name] = remove_small_blobs(opened, min_area=min_area)
    return cleaned


def evaluate_single_view_health(image_bgr, species=None, view=None):
    thresholds = get_thresholds(species, view)

    bg_color = get_background_reference_color(image_bgr)

    img = resize_with_padding(image_bgr, WORK_SIZE)
    leaf_mask = segment_leaf(img)

    leaf_silhouette_px = int(cv2.countNonZero(leaf_mask))
    if leaf_silhouette_px == 0:
        return None

    holes_mask = detect_holes(img, leaf_mask, bg_color)
    hole_px = int(cv2.countNonZero(holes_mask))

    tissue_mask = cv2.bitwise_and(leaf_mask, cv2.bitwise_not(holes_mask))
    tissue_px = int(cv2.countNonZero(tissue_mask))
    if tissue_px == 0:
        return None

    masks = build_health_masks(img, tissue_mask, thresholds)
    masks = clean_health_masks(masks)
    counts = {name: int(cv2.countNonZero(m)) for name, m in masks.items()}
    damage_px = counts["yellow"] + counts["brown"] + counts["black"] + counts["pale"]
    unclassified_px = max(0, tissue_px - counts["healthy"] - damage_px)

    tissue_quality_pct = round(counts["healthy"] / tissue_px * 100, 2)
    hole_pct = round(hole_px / leaf_silhouette_px * 100, 2)
    margin_loss_pct = estimate_margin_loss_pct(leaf_mask, species=species)

    final_health_pct = round(
        tissue_quality_pct * (1 - hole_pct / 100) * (1 - margin_loss_pct / 100), 2
    )

    return {
        "leaf_area_px": leaf_silhouette_px,
        "tissue_quality_pct": tissue_quality_pct,
        "yellow_pct": round(counts["yellow"] / tissue_px * 100, 2),
        "brown_pct": round(counts["brown"] / tissue_px * 100, 2),
        "black_pct": round(counts["black"] / tissue_px * 100, 2),
        "pale_pct": round(counts["pale"] / tissue_px * 100, 2),
        "unclassified_pct": round(unclassified_px / tissue_px * 100, 2),
        "hole_pct": hole_pct,
        "margin_loss_pct": margin_loss_pct,
        "final_health_pct": final_health_pct,
    }


def evaluate_full_leaf_health(top_bgr, bottom_bgr, species=None):
    top_report = evaluate_single_view_health(top_bgr, species=species, view="top") if top_bgr is not None else None
    bottom_report = evaluate_single_view_health(bottom_bgr, species=species, view="bottom") if bottom_bgr is not None else None

    if top_report is None and bottom_report is None:
        return None

    if top_report is None:
        combined_pct = bottom_report["final_health_pct"]
    elif bottom_report is None:
        combined_pct = top_report["final_health_pct"]
    else:
        top_area = top_report["leaf_area_px"]
        bottom_area = bottom_report["leaf_area_px"]
        combined_pct = round(
            (top_report["final_health_pct"] * top_area + bottom_report["final_health_pct"] * bottom_area)
            / (top_area + bottom_area),
            2,
        )

    return {
        "top": top_report,
        "bottom": bottom_report,
        "combined_final_health_pct": combined_pct,
    }
'''

In [ ]:
module_out_path = os.path.join(HEALTH_RESULTS_PATH, 'health_rules.py')
with open(module_out_path, 'w') as f:
    f.write(health_rules_source)

In [ ]:
print(f'✅ Exported: {module_out_path}')
print(f'   Species colour-calibrated : {list(SPECIES_THRESHOLD_OVERRIDES.keys())}')
print(f'   Species shape-calibrated  : {list(SPECIES_SHAPE_TEMPLATES_B64.keys())}')

## Cell 15 — Sanity Test the Exported Module

In [ ]:
import importlib.util

In [ ]:
spec = importlib.util.spec_from_file_location('health_rules', module_out_path)
health_rules = importlib.util.module_from_spec(spec)
spec.loader.exec_module(health_rules)

In [ ]:
if sample_species is not None and sample_imgs:
    notebook_result = evaluate_single_view_health(sample_bgr, species=sample_species, view=sample_view)
    module_result = health_rules.evaluate_single_view_health(sample_bgr, species=sample_species, view=sample_view)

    if notebook_result and module_result:
        match = notebook_result['final_health_pct'] == module_result['final_health_pct']
        print(f"  Notebook final health % : {notebook_result['final_health_pct']}")
        print(f"  Module   final health % : {module_result['final_health_pct']}")
        print('  ✅ MATCH — health_rules.py is ready to deploy!' if match else
              '  ⚠️  MISMATCH — re-run Cell 14 to re-export, then retest.')
else:
    print('⚠️  No sample image available to sanity-check against.')

## Cell 16 — Summary

In [ ]:
print('📊 HEALTH EVALUATION SUMMARY')
print('=' * 55)
if len(results_df):
    print(f'  Physical leaves evaluated : {len(results_df)}')
    print(f'  Species covered           : {results_df["species"].nunique()}')
    print(f'  Mean FINAL health %       : {results_df["combined_final_pct"].mean():.1f}%')
else:
    print('  No results yet -- run Cells 11-12 once HEALTH_RAW_PATH has data.')
print('=' * 55)
print(f'📁 CSV    : {os.path.join(HEALTH_RESULTS_PATH, "leaf_health_report.csv")}')
print(f'📁 Module : {os.path.join(HEALTH_RESULTS_PATH, "health_rules.py")}')
print('📋 Next step: calibrate_species(name) per species, paste threshold AND')
print('             template results into Cell 3, re-run Cell 14 to re-export.')